# Lab 09: INT4, quantised by you

Quantise Qwen3-0.6B to W4A16 with LLM Compressor, score the original and the result with lm_eval against the running server, then measure the prefill penalty. The quantisation step runs on the T4; expect ten to twenty minutes for it.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: the quantisation and evaluation tools.
%pip install -q "llmcompressor==0.14.0.1" "lm_eval[api]==0.4.13" 2>&1 | tail -2
import llmcompressor, lm_eval; print("llmcompressor", llmcompressor.__version__, "· lm_eval", lm_eval.__version__)

In [ ]:
# Cell 3: helpers used by every lab: start the server in the background, wait for it, talk to it.
import json, re, subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-0.6B"
URL = "http://127.0.0.1:8000"

def serve(extra_args=(), log_path="/kaggle/working/serve.log", timeout_s=900):
    args = ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", *extra_args]   # the T4 has no bfloat16
    log = open(log_path, "w")
    t0 = time.time()
    proc = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT)
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError("vllm serve exited; see " + log_path)
        try:
            urllib.request.urlopen(URL + "/health", timeout=2); return proc, round(time.time() - t0, 1)
        except Exception:
            time.sleep(0.5)
    raise RuntimeError("server not ready in time")

def stop(proc):
    proc.terminate(); proc.wait()

def log_lines(path, patterns):
    text = open(path).read(); out = {}
    for name, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            line = re.sub(r"^\(EngineCore[^)]*\) ", "", m.group(0)); line = re.sub(r"^INFO \d\d-\d\d \d\d:\d\d:\d\d \[[^\]]+\] ", "", line); out[name] = line.strip()
    return out

STARTUP = {"dtype": r"INFO[^\n]*dtype=torch\.[a-z0-9]+", "backend": r"Using [A-Z_]+ attention backend[^\n]*", "weights": r"Loading weights took [^\n]*",
           "model": r"Model loading took [^\n]*", "kv_memory": r"Available KV cache memory: [^\n]*", "kv_size": r"GPU KV cache size: [^\n]*", "graphs": r"Graph capturing finished[^\n]*"}

def chat(messages, **params):
    body = json.dumps({"model": MODEL, "messages": messages, **params}).encode()
    req = urllib.request.Request(URL + "/v1/chat/completions", data=body, headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=300).read())
print("helpers ready")

In [ ]:
# Cell 5: the benchmark call.
def bench_quiet(rate, num_prompts, input_len=1024, output_len=128, max_concurrency=None, model=None):
    args = ["vllm", "bench", "serve", "--model", model or MODEL, "--tokenizer", MODEL, "--port", "8000", "--dataset-name", "random", "--random-input-len", str(input_len),
            "--random-output-len", str(output_len), "--num-prompts", str(num_prompts), "--request-rate", str(rate),
            "--metric-percentiles", "50,95,99", "--percentile-metrics", "ttft,tpot,itl,e2el", "--ignore-eos", "--seed", "0"]
    if max_concurrency: args += ["--max-concurrency", str(max_concurrency)]
    out = subprocess.run(args, capture_output=True, text=True); text = out.stdout; i = text.find("Serving Benchmark Result")
    res = {}
    for line in text.splitlines():
        m = re.match(r'^(.*?):\s+([0-9.]+)\s*$', line)
        if m: res[m.group(1).strip()] = float(m.group(2))
    return res
def row(label, r):
    g = lambda k: r.get(k, float("nan"))
    return f"{label:<28} req/s {g('Request throughput (req/s)'):5.2f}  out tok/s {g('Output token throughput (tok/s)'):7.1f}  TTFT p50 {g('P50 TTFT (ms)'):6.0f} p99 {g('P99 TTFT (ms)'):6.0f} ms  TPOT p50 {g('P50 TPOT (ms)'):5.1f} p99 {g('P99 TPOT (ms)'):5.1f} ms  peak conc {g('Maximum request concurrency'):3.0f}"


In [ ]:
# Cell 6: quantise. GPTQ, 4-bit weights with 16-bit activations, 128 calibration samples from wikitext.
import os, time, torch
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
OUT = "/kaggle/working/Qwen3-0.6B-W4A16"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="cuda", dtype=torch.float16)
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train").filter(lambda x: len(x["text"]) > 400).shuffle(seed=42).select(range(128))
ds = ds.map(lambda x: tok(x["text"], max_length=2048, truncation=True, add_special_tokens=False), remove_columns=ds.column_names)
t0 = time.time()
oneshot(model=model, tokenizer=tok, dataset=ds, recipe=GPTQModifier(scheme="W4A16", targets="Linear", ignore=["lm_head"]), output_dir=OUT, max_seq_length=2048, num_calibration_samples=128, pipeline="basic")
size = sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT) if f.endswith(".safetensors"))
print(f"took {time.time() - t0:.0f} s; checkpoint {size / 1e9:.2f} GB against 1.50 GB")
del model; torch.cuda.empty_cache()

In [ ]:
# Cell 7: serve each one, score it on 100 HellaSwag examples, measure it.
def lm_eval_run(model_name, limit=100):
    args = ["lm_eval", "--model", "local-completions", "--model_args", f"model={model_name},base_url={URL}/v1/completions,num_concurrent=8,max_retries=2,tokenized_requests=False", "--tasks", "hellaswag", "--limit", str(limit), "--batch_size", "8"]
    out = subprocess.run(args, capture_output=True, text=True); text = out.stdout + out.stderr
    acc = re.findall(r"acc_norm\s*\|[^|]*\|\s*([0-9.]+)", text); return acc[0] if acc else text[-600:]
for label, model in (("fp16 original", "Qwen/Qwen3-0.6B"), ("W4A16 quantised", OUT)):
    MODEL = model
    proc, ready_s = serve(["--max-model-len", "4096", "--gpu-memory-utilization", "0.8"]); print(f"{label}: ready after {ready_s} s")   # the notebook process keeps about 2 GiB after quantising, so the server asks for less
    print("  " + log_lines("/kaggle/working/serve.log", {"q": r"quantization=[a-z0-9_]+"}).get("q", "quantization=none"))
    print(f"  hellaswag acc_norm {lm_eval_run(model)}")
    print("  " + row("1 at a time, 1,024 in", bench_quiet(rate="inf", num_prompts=16, input_len=1024, max_concurrency=1, model=model)))
    print("  " + row("32 concurrent, 1,024 in", bench_quiet(rate="inf", num_prompts=64, input_len=1024, max_concurrency=32, model=model)))
    stop(proc)
MODEL = "Qwen/Qwen3-0.6B"
